# HumidClimatologyEngine — Guided Tutorial

## The journey from ERA5-Land to a distribution-aware moisture climatology

This notebook teaches the project through small, executable examples. It does not require the full 1981–2020 dataset.

**Learning route:** motivation → data acquisition → leap-day rule → physics → nonlinearity → Welford → joint Gaussian model → Monte Carlo → convergence → production workflow.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import importlib.util

CORE = Path('../src/moisture_climatology_v6.py').resolve()
spec = importlib.util.spec_from_file_location('humid_engine', CORE)
engine = importlib.util.module_from_spec(spec)
spec.loader.exec_module(engine)
print(CORE)


## 1. The central idea

Relative humidity is nonlinear. Therefore, calculating RH from climatological means is not generally equivalent to averaging RH computed from the underlying joint population. HumidClimatologyEngine therefore models `(T, Td, logP)` first and applies the physics to samples afterward.

## 2. ERA5-Land acquisition

The repository includes `scripts/download_era5land_daily_statistics.py`.

Example for the historical 2021 data:

```bash
python scripts/download_era5land_daily_statistics.py \
  --years 2021 \
  --variables 2m_temperature 2m_dewpoint_temperature surface_pressure \
  --statistics daily_mean daily_minimum daily_maximum \
  --out-dir ./data/raw
```

Your legacy manually calculated 2021 statistics can remain as a separate provenance-controlled archive.

## 3. Leap-day convention

Project rule:

```text
DOY 59 = reserved
DOY 60 = Feb 28 + Feb 29 pooled
DOY 61 = Mar 1
```

In [ ]:
examples = pd.Series({
    '1984-02-28': engine.get_clim_doy(59, 1984),
    '1984-02-29': engine.get_clim_doy(60, 1984),
    '1984-03-01': engine.get_clim_doy(61, 1984),
    '1985-02-28': engine.get_clim_doy(59, 1985),
    '1985-03-01': engine.get_clim_doy(60, 1985),
})
examples


## 4. One physical sample

Use `T=25°C`, `Td=18°C`, `P=1005 hPa`.

In [ ]:
T=np.array([25.],dtype=np.float32)
Td=np.array([18.],dtype=np.float32)
P=np.array([1005.],dtype=np.float32)
e=engine.calculate_vapor_pressure(Td)
rh=engine.calculate_rh_from_td(T,Td)
r=engine.calculate_mixing_ratio(e,P)
q=engine.calculate_specific_humidity(r)
pd.DataFrame({'T_C':T,'Td_C':Td,'P_hPa':P,'e_hPa':e,'RH_%':rh,'r_kgkg':r,'q_kgkg':q})


## 5. Nonlinearity experiment

We compare `RH(mean T, mean Td)` with `mean(RH(T,Td))`.

In [ ]:
rng=np.random.default_rng(42)
n=10000
T=rng.normal(20,5,n).astype(np.float32)
Td=(T-np.abs(rng.normal(5,2,n))).astype(np.float32)
rh_samples=engine.calculate_rh_from_td(T,Td)
rh_of_means=float(engine.calculate_rh_from_td(np.array([T.mean()],dtype=np.float32),np.array([Td.mean()],dtype=np.float32))[0])
mean_rh=float(np.nanmean(rh_samples))
print('RH(mean T, mean Td):',rh_of_means)
print('mean(RH samples):   ',mean_rh)
print('difference:          ',mean_rh-rh_of_means)


## 6. Welford intuition

Welford updates mean and variance online without storing the complete historical series.

In [ ]:
x=rng.normal(10,3,100000)
mean=0.; M2=0.; n=0
for value in x:
    n+=1
    delta=value-mean
    mean += delta/n
    M2 += delta*(value-mean)
print('Welford mean:',mean)
print('NumPy mean:  ',x.mean())
print('Welford var: ',M2/(n-1))
print('NumPy var:   ',x.var(ddof=1))


## 7. Synthetic `(T, Td, logP)` model

The next cell creates a known correlation matrix and verifies that it is positive definite before sampling.

In [ ]:
mu=np.array([20.,13.,np.log(1010.)])
sigma=np.array([5.,4.,0.03])
R=np.array([[1,.75,.15],[.75,1,.20],[.15,.20,1.]])
cov=np.diag(sigma)@R@np.diag(sigma)
print('eigenvalues:',np.linalg.eigvalsh(R))
s= rng.multivariate_normal(mu,cov,size=20000)
T,Td,logP=s.T
P=np.exp(logP)


## 8. Propagate the population through the moisture physics

In [ ]:
T=T.astype(np.float32); Td=Td.astype(np.float32); P=P.astype(np.float32)
RH=engine.calculate_rh_from_td(T,Td)
e=engine.calculate_vapor_pressure(Td)
r=engine.calculate_mixing_ratio(e,P)
q=engine.calculate_specific_humidity(r)
pd.DataFrame({'RH':RH,'e_hPa':e,'r_kgkg':r,'q_kgkg':q}).describe().T


## 9. Plot the transformed RH distribution

In [ ]:
plt.figure(figsize=(8,4.5))
plt.hist(RH[np.isfinite(RH)],bins=50)
plt.xlabel('Relative humidity (%)')
plt.ylabel('Count')
plt.title('Synthetic RH distribution')
plt.tight_layout()
plt.show()


## 10. Supersaturation diagnostic

Before clipping RH, identify samples with `RH_raw > 100%`.

In [ ]:
esT=engine.saturation_vapor_pressure(T)
esTd=engine.saturation_vapor_pressure(Td)
rh_raw=100*esTd/esT
print('supersaturation fraction:',np.mean(rh_raw>100))


## 11. Convergence

The production sample count should be chosen from convergence rather than from habit. Compare 500, 1000, 2000, 5000 and 10000.

In [ ]:
def summary(N, seed=20260821):
    local=np.random.default_rng(seed)
    z=local.multivariate_normal(mu,cov,size=N)
    t,td,lp=z.T
    p=np.exp(lp).astype(np.float32)
    t=t.astype(np.float32); td=td.astype(np.float32)
    rh=engine.calculate_rh_from_td(t,td)
    ee=engine.calculate_vapor_pressure(td)
    rr=engine.calculate_mixing_ratio(ee,p)
    qq=engine.calculate_specific_humidity(rr)
    return {'N':N,'mean_RH':float(np.nanmean(rh)),'std_RH':float(np.nanstd(rh,ddof=1)),
            'mean_q':float(np.nanmean(qq))}
pd.DataFrame([summary(n) for n in [500,1000,2000,5000,10000]])


## 12. Production checklist

Before a full 1981–2020 run:

- authenticate with CDS;
- verify input completeness and coordinate alignment;
- run the mandatory tests;
- run Monte Carlo convergence experiments;
- preserve annual/day checkpoints;
- archive the configuration, Git commit and input checksums;
- validate physical output ranges after NetCDF creation.


## Final lesson

The key sequence is:

```text
joint drivers
   ↓
joint statistical model
   ↓
Monte Carlo samples
   ↓
nonlinear physics
   ↓
distributional summaries
```

That ordering is the scientific heart of HumidClimatologyEngine.